# E3 — Cross-dataset: generalisation on SH17 (out-of-distribution)
Takes the two detectors ALREADY TRAINED on PPE-Unified (no retraining) and evaluates them on SH17_crosstest — real manufacturing-floor imagery, with the 4 overlapping classes remapped to PPE-Unified IDs: helmet(0), vest(2), gloves(4), mask(10).

**One-time setup:** upload the `Data Set/SH17_crosstest` folder to Kaggle as a **PRIVATE** dataset, suggested name `sh17-crosstest` (SH17 is CC-BY-NC-SA — do NOT make it public).

**Add Input (3 items):** ① dataset `sh17-crosstest` · ② notebook 01 output `notebook074c5c1ed8` — **PIN VERSION 2** (the real 12 h run; V5 contains quick-test weights!) · ③ notebook 03 output `notebook75023f5e97` (the completed version).

**Session:** GPU T4 x2 · Internet ON (for pip only). Estimated ~30-60 min.

In [ ]:
# ==================== CONFIG (E3) ====================
import os, glob, json, yaml, shutil, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
# RESUME: load the E3 checkpoint from a previous session's output (if this notebook's earlier output was Added as Input)
import glob as _g
for _st in _g.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith("e3_") for k in _s):
        shutil.copy(_st, STATE_PATH); print("E3 CHECKPOINT LOADED FROM:", _st); break
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)
def done(k): return k in STATE
def mark(k, v): STATE[k] = v; save_state()

In [ ]:
%pip install -q ultralytics
import torch; assert torch.cuda.is_available(), "Enable GPU T4!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")

## 1. Locate the weights (with a GUARD against quick-test builds) and the SH17_crosstest dataset

In [ ]:
def epochs_of(rcsv):
    return sum(1 for _ in open(rcsv)) - 1 if os.path.exists(rcsv) else 0

def find_notebook_best(slug, run_name, min_epochs=20):
    """Tim best.pt trong output notebook; tra None neu khong co hoac la ban quick-test."""
    for cand in [f"/kaggle/input/notebooks/quocdatlam/{slug}", f"/kaggle/input/{slug}"]:
        w = os.path.join(cand, "runs", run_name, "weights", "best.pt")
        if os.path.exists(w):
            n = epochs_of(os.path.join(cand, "runs", run_name, "results.csv"))
            if n >= min_epochs:
                print(f"{run_name}: best.pt OK ({n} epochs, tu output notebook) ->", w); return w
            print(f"{run_name}: output notebook la ban quick-test ({n} epochs) -> thu fallback dataset...")
    return None

def find_dataset_best(keyword):
    """Fallback: best.pt do user upload lam dataset (khong tin notebook mount nua)."""
    cands = [p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True)
             if keyword in p.lower() and "best" in os.path.basename(p).lower()
             and "/notebooks/" not in p and os.path.getsize(p) > 5e6]
    return sorted(cands)[0] if cands else None

YOLO_W = find_notebook_best("notebook074c5c1ed8", "yolo_final") or find_dataset_best("yolo")
assert YOLO_W, ("Khong co weights YOLO full-run! Tai runs/yolo_final/weights/best.pt tu OUTPUT VERSION 2 "
    "cua notebook 01 ve may, upload lam dataset private (vd 'yolo26s-ppe-best') roi Add Input vao day.")
print("YOLO26s weights:", YOLO_W)

RT_W = find_notebook_best("notebook75023f5e97", "rtdetr_final") or find_dataset_best("rtdetr")
assert RT_W, "Khong co weights RT-DETR full-run! (output notebook 03 version hoan thanh phai co runs/rtdetr_final)"
print("RT-DETR weights:", RT_W)

# --- tim SH17_crosstest ---
cands = [p for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True) if "sh17" in p.lower()]
assert cands, "Chua Add Input dataset sh17-crosstest (hoac ten khong chua 'sh17')!"
SH_ROOT = os.path.dirname(cands[0])
print("SH17_crosstest:", SH_ROOT)

# --- dung data yaml tuyet doi, tro val vao thu muc anh cua crosstest ---
import yaml
y0 = yaml.safe_load(open(cands[0]))
names = y0["names"]
img_dir = None
for k in ["test", "val", "train"]:
    v = y0.get(k)
    if v:
        c2 = v if os.path.isabs(str(v)) else os.path.join(SH_ROOT, str(v))
        if os.path.exists(c2): img_dir = c2; break
if img_dir is None:
    hits = sorted(glob.glob(f"{SH_ROOT}/**/images", recursive=True))
    assert hits, "Khong tim thay thu muc images trong sh17-crosstest"
    img_dir = hits[0]
FIXED = f"{WORK}/sh17_eval.yaml"
yaml.safe_dump({"path": SH_ROOT, "train": img_dir, "val": img_dir,
                "names": names, "nc": len(names)}, open(FIXED, "w"))
n_img = len(glob.glob(f"{img_dir}/*"))
print(f"Anh danh gia: {n_img} | 12 lop (chi 4 lop co nhan: helmet/vest/gloves/mask)")

## 2. Evaluate both detectors on SH17 (4 mapped classes) against in-distribution performance

In [ ]:
from ultralytics import YOLO, RTDETR
import pandas as pd
MAPPED = {0: "helmet", 2: "vest", 4: "gloves", 10: "mask"}
# In-distribution AP@50 (PPE-Unified TEST) for exactly these 4 classes — taken from the existing per-class table:
IN_DIST = {"YOLO26s": {"helmet": 0.9521, "vest": 0.9072, "gloves": 0.8027, "mask": 0.9392},
           "RT-DETR-L": {"helmet": 0.9464, "vest": 0.9183, "gloves": 0.8003, "mask": 0.9124}}

rows = []
for label, (cls, w) in {"YOLO26s": (YOLO, YOLO_W), "RT-DETR-L": (RTDETR, RT_W)}.items():
    key = f"e3_{label}"
    if done(key): rows.append(STATE[key]); print("skip:", STATE[key]); continue
    t0 = time.time()
    m = cls(w)
    v = m.val(data=FIXED, split="val", batch=4, imgsz=640, verbose=False)
    per = {}
    for idx, ap in zip(v.box.ap_class_index, v.box.ap50):
        if int(idx) in MAPPED: per[MAPPED[int(idx)]] = round(float(ap), 4)
    ood = round(sum(per.values()) / max(len(per), 1), 4)
    ind = round(sum(IN_DIST[label].values()) / 4, 4)
    row = {"model": label, **{f"AP50_{k}": v_ for k, v_ in per.items()},
           "mAP50_4cls_SH17": ood, "mAP50_4cls_inDist": ind,
           "drop": round(ind - ood, 4), "min": round((time.time() - t0) / 60, 1)}
    rows.append(row); mark(key, row); print(row)
    del m; import gc, torch; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(rows); display(df)
df.to_csv(f"{WORK}/results/e3_sh17.csv", index=False)
print("\nNote: SH17 has no no_* violation classes, so only the 4 compliant classes are compared;")
print("the drop is the domain gap from in-distribution to a real manufacturing environment.")

In [ ]:
# ==================== CLEAN UP OUTPUT: make sure no dataset is bundled ====================
# E3 reads SH17 straight from /kaggle/input (nothing is copied into working), but this cell
# does a final sweep: keep only results/, state.json, sh17_eval.yaml and runs/ (if small).
KEEP = {"results", "state.json", "sh17_eval.yaml", "runs"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

def du(path):
    total = 0
    for r, _, fs in os.walk(path):
        total += sum(os.path.getsize(os.path.join(r, f)) for f in fs)
    return total

runs = f"{WORK}/runs"
if os.path.isdir(runs) and du(runs) > 200e6:   # runs/ should only hold a few small val plots
    shutil.rmtree(runs, ignore_errors=True); print("runs/ too large -> deleted")
print("Final output:", sorted(os.listdir(WORK)),
      f"| total {du(WORK)/1e6:.1f} MB (no dataset included)")